# Simple preprocessing + log target
Median imputation -> log1p for six selected numeric features -> StandardScaler; missing label -> one-hot for categorical features.
Train/test splitting happens first. Fit preprocessing on train only. Use log1p on SalePrice through TransformedTargetRegressor; all predictions and metrics use original prices.
No custom transformers, clipping or automatic feature power transforms.

In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, HuberRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

## Load and split
Use the same columns and split as main.ipynb for comparison.

In [14]:
DATA_PATH = "data/train-house-prices-advanced-regression-techniques.csv"
house_df = pd.read_csv(DATA_PATH).drop(columns=["Id", "Alley", "PoolQC", "Fence", "MiscFeature"])
train_df, test_df = train_test_split(house_df, test_size=0.25, random_state=42)
y_train = train_df.pop("SalePrice").to_numpy()
y_test = test_df.pop("SalePrice").to_numpy()
num_cols = train_df.select_dtypes(include="number").columns.tolist()
cat_cols = train_df.select_dtypes(exclude="number").columns.tolist()

## Fill missing values, log selected features, and scale
Median imputation -> log1p for GrLivArea, GarageArea, TotalBsmtSF, 1stFlrSF, TotRmsAbvGrd and YearBuilt -> StandardScaler.
log1p keeps zeros valid. YearBuilt is transformed directly; no age feature is introduced.


In [15]:
imputer = SimpleImputer(strategy="median")
train_num_features = imputer.fit_transform(train_df[num_cols])
test_num_features = imputer.transform(test_df[num_cols])

log_cols = ["GrLivArea", "GarageArea", "TotalBsmtSF", "1stFlrSF", "TotRmsAbvGrd", "YearBuilt"]
log_indices = [num_cols.index(col) for col in log_cols]
train_num_features[:, log_indices] = np.log1p(train_num_features[:, log_indices])
test_num_features[:, log_indices] = np.log1p(test_num_features[:, log_indices])

scaler = StandardScaler()
train_num_features = scaler.fit_transform(train_num_features)
test_num_features = scaler.transform(test_num_features)

## One-hot encoding and concatenate
Drop the reference category to reduce exact collinearity for linear models. Unseen categories are handled by the encoder. The original train_df/test_df remain raw for cross-validation below.

In [16]:
encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
train_cat_features = encoder.fit_transform(train_df[cat_cols].fillna("none"))
test_cat_features = encoder.transform(test_df[cat_cols].fillna("none"))
X_train = np.hstack([train_num_features, train_cat_features])
X_test = np.hstack([test_num_features, test_cat_features])
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (1095, 243) Test: (365, 243)


c:\Users\ADMIN\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [9, 13, 28] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## Train and evaluate on original prices
Lasso alpha is smaller because it now operates on log prices. Parameters below are starting points, not tuned using the test set. MLP also receives a standardized log target for easier optimization.

In [17]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1),
    "Lasso": Lasso(alpha=0.001, max_iter=10000),
    "Huber": HuberRegressor(alpha=0.01, max_iter=5000, tol=0.0001),
    "MLP": MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1, early_stopping=True,
                        n_iter_no_change=20, max_iter=1000, random_state=42),
}
fitted_models, results = {}, []
for name, estimator in models.items():
    model = TransformedTargetRegressor(regressor=estimator, func=np.log1p, inverse_func=np.expm1)
    if name == "MLP":
        model.regressor = TransformedTargetRegressor(regressor=estimator, transformer=StandardScaler())
    model.fit(X_train, y_train)
    fitted_models[name] = model
    train_pred, test_pred = model.predict(X_train), model.predict(X_test)
    results.append({"Model": name,
                    "Train_RMSE": np.sqrt(mean_squared_error(y_train, train_pred)),
                    "Test_RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
                    "Train_R2": r2_score(y_train, train_pred), "Test_R2": r2_score(y_test, test_pred)})
df_results = pd.DataFrame(results).set_index("Model")
df_results.round(3)

,Train_RMSE,Test_RMSE,Train_R2,Test_R2
Model,,,,
Linear Regression,17543.323,24049.104,0.949,0.917
Ridge,20409.390,23948.705,0.931,0.918
Lasso,26408.023,25862.909,0.885,0.905
Huber,21834.157,24053.405,0.921,0.917
MLP,15396.514,29110.860,0.961,0.879


In [18]:
from sklearn.base import clone
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
poly_scaler = StandardScaler()
train_poly = poly_scaler.fit_transform(poly.fit_transform(train_num_features))
test_poly = poly_scaler.transform(poly.transform(test_num_features))
X_train_poly = np.hstack([train_poly, train_cat_features])
X_test_poly = np.hstack([test_poly, test_cat_features])

poly_models, poly_results = {}, []
for name, baseline in fitted_models.items():
    model = clone(baseline).fit(X_train_poly, y_train)
    poly_models[name] = model
    train_pred, test_pred = model.predict(X_train_poly), model.predict(X_test_poly)
    poly_results.append({"Model": name,
                         "Train_RMSE": np.sqrt(mean_squared_error(y_train, train_pred)),
                         "Test_RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
                         "Train_R2": r2_score(y_train, train_pred), "Test_R2": r2_score(y_test, test_pred)})
df_poly_results = pd.DataFrame(poly_results).set_index("Model")
print(f"Features: {X_train.shape[1]} -> {X_train_poly.shape[1]}")
df_poly_results.join(df_results[["Test_RMSE", "Test_R2"]].add_prefix("Baseline_")).round(3)

Features: 243 -> 909


STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html


,Train_RMSE,Test_RMSE,Train_R2,Test_R2,Baseline_Test_RMSE,Baseline_Test_R2
Model,,,,,,
Linear Regression,8336.320,1.834745e+07,0.989,-48052.395,24049.104,0.917
Ridge,9502.704,4.160440e+04,0.985,0.753,23948.705,0.918
Lasso,15282.814,2.656815e+04,0.962,0.899,25862.909,0.905
Huber,12953.749,2.400106e+07,0.972,-82229.601,24053.405,0.917
MLP,25538.157,7.878967e+04,0.893,0.114,29110.860,0.879
